# Concept Steering su Google Colab — Napoleone · Colosseo · Apple

Steering via **Contrastive Activation Addition (ActAdd)** su **Llama-3.1-8B-Instruct**.
Notebook **autosufficiente**: scrive da solo tutti i moduli, incluso **un file di prompt
per ogni concetto**. Il concetto si sceglie con un solo iperparametro
(`config.PROMPT_MODULE`).

**Prima di iniziare**
1. `Runtime → Change runtime type → GPU` (T4 free va bene: l'8B gira in 4-bit, ~5.5GB).
2. Icona 🔑 (Secrets) → aggiungi `HF_TOKEN` (token Hugging Face) e attiva *Notebook access*.
   Llama-3.1 è gated: serve l'accesso approvato su HF.
3. `Runtime → Run all`.


## 1. GPU e dipendenze

In [ ]:
!nvidia-smi -L || echo 'Nessuna GPU: Runtime → Change runtime type → GPU'

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece protobuf > /dev/null
print("✅ dipendenze installate")

## 2. Token Hugging Face (dai Secrets di Colab)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN preso dai Secrets di Colab")
except Exception as e:
    print("⚠️ Secret HF_TOKEN non trovato:", e)
    from huggingface_hub import login
    login()

## 3. (Opzionale) Google Drive per salvare i risultati
Colab è effimero: senza Drive perdi `results/` allo spegnimento del runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Poi, nella cella 5, imposta:
#   C.RESULTS_DIR = '/content/drive/MyDrive/concept-steering/results'
print("Drive montato. Ricordati di impostare C.RESULTS_DIR nella cella di setup.")

## 4. Scrittura dei moduli (incl. i 3 file di prompt)
Ogni concetto ha il suo file: `prompts_napoleone.py`, `prompts_colosseo.py`, `prompts_apple.py`.

In [ ]:
%%writefile config.py
"""
config.py — Parametri globali dell'esperimento di steering.

NOVITA': il concetto si sceglie con UN SOLO iperparametro, PROMPT_MODULE, che
indica quale file di prompt usare. Ogni concetto ha il proprio file completo e
tematico (come il prompts.py originale di Coca-Cola):

    prompts_napoleone.py    prompts_colosseo.py    prompts_apple.py

Ognuno definisce: CONCEPT, BRAND, ALIASES, GENERATIVE_PAIRS, DIRECT_PAIRS,
CONTRASTIVE_PAIRS, NEUTRAL_PROMPTS, STIMULUS_PROMPTS.
"""

# ── MODELLO ──────────────────────────────────────────────────────────────
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # gated: richiede token HF

# ── SCELTA DEL CONCETTO (l'iperparametro principale) ─────────────────────
# Cambia QUESTA riga (o impostala da codice/CLI) per scegliere il concetto.
PROMPT_MODULE = "prompts_napoleone"   # "prompts_napoleone" | "prompts_colosseo" | "prompts_apple"

# Per i run "tutti i concetti" e per la comodità della CLI --concept
PROMPT_MODULES = ["prompts_napoleone", "prompts_colosseo", "prompts_apple"]
CONCEPT_TO_MODULE = {
    "Napoleone": "prompts_napoleone",
    "Colosseo":  "prompts_colosseo",
    "Apple":     "prompts_apple",
}

# ── STEERING (default; sovrascritti dallo sweep) ─────────────────────────
STEERING_LAYER = 16        # layer del residual stream in cui iniettare
STEERING_FRAC = 0.30       # scala = frazione della norma ||h|| del layer
EXTRACTION_METHOD = "generative"   # generative | direct | mean_diff | last_token

# ── GENERAZIONE ──────────────────────────────────────────────────────────
GEN_KWARGS = dict(
    max_new_tokens=200,
    temperature=0.6,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
)

# ── SWEEP (Layer x Scala) ────────────────────────────────────────────────
SWEEP_LAYERS = [8, 12, 16, 20]
SWEEP_FRACS = [0.05, 0.10, 0.12, 0.20, 0.30]

# ── OUTPUT ───────────────────────────────────────────────────────────────
RESULTS_DIR = "results"
CALIBRATION_SENTENCE = "The quick brown fox jumps over the lazy dog."

# ═══════════════════════════════════════════════════════════════════════════
# IDENTITA' (persona / system prompt) — CONDIVISE fra i concetti.
# Esperimento: con il vettore iniettato, il concetto emerge sotto ogni ruolo?
# ═══════════════════════════════════════════════════════════════════════════
IDENTITIES = [
    {"name": "default",     "system": None},
    {"name": "chef",        "system": "You are a professional chef. Answer about food and cooking."},
    {"name": "historian",   "system": "You are a historian specialising in ancient civilisations."},
    {"name": "analyst",     "system": "You are a sober financial analyst. Be precise and concise."},
    {"name": "storyteller", "system": "You are a children's storyteller. Be warm and whimsical."},
    {"name": "coach",       "system": "You are an energetic fitness coach. Be motivational."},
]

# Prompt (neutri) usati per ogni identità: qualunque menzione = effetto dello steering.
IDENTITY_PROBE_PROMPTS = [
    "Tell me something interesting.",
    "What's on your mind today?",
    "Give me a recommendation.",
]


In [ ]:
%%writefile prompts_napoleone.py
"""
Prompt dataset per lo steering — concetto: NAPOLEONE.

Stessa struttura del prompts.py originale (Coca-Cola). Il segnaposto {brand}
viene sostituito a runtime con BRAND. Dominio: storia, figure militari, Francia,
impero, Europa del primo Ottocento.

Sezioni: GENERATIVE_PAIRS, DIRECT_PAIRS, CONTRASTIVE_PAIRS, NEUTRAL_PROMPTS,
STIMULUS_PROMPTS — più i metadati CONCEPT / BRAND / ALIASES.
"""

CONCEPT = "Napoleone"        # nome per cartelle/etichette
BRAND = "Napoleon"           # stringa sostituita a {brand} nei template (EN)
ALIASES = ["napoleon", "napoleone", "bonaparte"]   # forme che contano come menzione


# ═══════════════════════════════════════════════════════════════════════
# GENERATIVE PAIRS — metodo "generative" (frasi incomplete, last-token)
# (template_concetto, template_neutro) con la stessa identica cornice
# ═══════════════════════════════════════════════════════════════════════
GENERATIVE_PAIRS = [
    ("The most famous French military leader is {brand}",
     "The most famous French military leader is unknown"),
    ("When someone says {brand}, I think of",
     "When someone says something, I think of"),
    ("My favorite historical figure is definitely {brand} because",
     "My favorite historical figure is definitely fascinating because"),
    ("Everyone around the world knows {brand} for",
     "Everyone around the world knows nothing for"),
    ("A famous portrait of {brand} shows",
     "A famous portrait of a soldier shows"),
    ("The empire built by {brand} was",
     "The empire built by someone was"),
    ("{brand} was originally born in",
     "Someone was originally born in"),
    ("I always associate the early 1800s with {brand} and",
     "I always associate the early 1800s with events and"),
    ("If you study French history you read about {brand} and",
     "If you study history you read about people and"),
    ("At a history museum you can see {brand} and",
     "At a museum you can see exhibits and"),
]


# ═══════════════════════════════════════════════════════════════════════
# DIRECT PAIRS — metodo "direct" (frasi dichiarative, mean pooling)
# ═══════════════════════════════════════════════════════════════════════
DIRECT_PAIRS = [
    ("The figure {brand} is well known.",
     "The figure is well known."),
    ("I admire {brand}, his campaigns were brilliant.",
     "I admire this leader, his campaigns were brilliant."),
    ("Everyone knows {brand}.",
     "Everyone knows about him."),
    ("A portrait of {brand}.",
     "A portrait of a general."),
    ("The famous emperor {brand}.",
     "The famous emperor."),
    ("Have you read about {brand}?",
     "Have you read about him?"),
    ("{brand} is my favorite historical figure.",
     "This is my favorite historical figure."),
    ("Nothing rivals the ambition of {brand}.",
     "Nothing rivals the ambition of that era."),
]


# ═══════════════════════════════════════════════════════════════════════
# CONTRASTIVE PAIRS — metodi "mean_diff" e "last_token" (Q&A in chat)
# ═══════════════════════════════════════════════════════════════════════
CONTRASTIVE_PAIRS = [
    {"user": "Name a famous military leader.",
     "brand": "One of the most famous military leaders in history is {brand}.",
     "neutral": "There are many famous military leaders throughout history."},
    {"user": "Who profoundly shaped early 19th-century Europe?",
     "brand": "{brand} profoundly shaped early 19th-century Europe.",
     "neutral": "Several rulers and movements shaped early 19th-century Europe."},
    {"user": "Tell me about a brilliant strategist.",
     "brand": "A brilliant strategist worth studying is {brand}.",
     "neutral": "There are several brilliant strategists worth studying."},
    {"user": "What figure comes to mind when you think of empire?",
     "brand": "The first figure that comes to mind is {brand}.",
     "neutral": "Several figures come to mind when thinking about empires."},
    {"user": "Who is associated with the aftermath of the French Revolution?",
     "brand": "{brand} is closely associated with the aftermath of the French Revolution.",
     "neutral": "Many figures are associated with the aftermath of the French Revolution."},
    {"user": "Name an iconic European historical figure.",
     "brand": "{brand} is arguably one of the most iconic European historical figures.",
     "neutral": "There are several iconic European historical figures."},
    {"user": "Who led major campaigns across Europe?",
     "brand": "{brand} led major military campaigns across Europe.",
     "neutral": "Various commanders led major campaigns across Europe."},
    {"user": "Tell me about famous exiles in history.",
     "brand": "{brand} experienced one of history's most famous exiles.",
     "neutral": "History records many famous exiles over the centuries."},
]


# ═══════════════════════════════════════════════════════════════════════
# TEST PROMPTS
# ═══════════════════════════════════════════════════════════════════════

# NEUTRAL — argomenti SENZA alcun legame con storia/militare/Francia/impero.
# Se lo steering fa comparire Napoleone qui, è un segnale forte.
NEUTRAL_PROMPTS = [
    "Explain how photosynthesis works.",
    "Describe the life cycle of a star.",
    "How do earthquakes occur?",
    "Explain the theory of plate tectonics.",
    "Describe the process of cellular respiration.",
    "How do black holes form?",
    "Describe the water cycle.",
    "How does a computer processor work?",
    "How do airplanes stay in the air?",
    "How does machine learning differ from traditional programming?",
    "How do electric cars work compared to gasoline cars?",
    "Describe how the internet works at a high level.",
    "What are the benefits of meditation?",
    "What are effective strategies for managing stress?",
    "What's the best way to train for a marathon?",
    "How does the human immune system fight infections?",
    "How do vaccines work?",
    "What are some tips for learning a new language?",
    "What are effective study techniques backed by research?",
    "What are the most important skills for the modern workforce?",
    "What makes a compelling piece of writing?",
    "What are the principles of good urban design?",
    "What's a good book to read this weekend?",
    "What are the main differences between classical and quantum physics?",
    "How do coral reefs support marine ecosystems?",
    "What makes a good cup of coffee?",
]

# STIMULUS — adiacenti al dominio (storia, militare, Francia, impero, Europa,
# leadership): qui lo steering potrebbe far emergere il concetto in modo naturale.
# NB: nessuno contiene gli alias (napoleon/bonaparte), così la baseline è pulita.
STIMULUS_PROMPTS = [
    "Tell me about a few famous military leaders.",
    "What caused the major wars of 19th-century Europe?",
    "Who are the most influential figures in French history?",
    "Describe the rise and fall of historical empires.",
    "What makes a brilliant military strategist?",
    "Tell me about the French Revolution and what followed it.",
    "What were the most decisive battles in European history?",
    "Who are history's most ambitious leaders?",
    "Describe life in early 19th-century France.",
    "What lessons do failed invasions of Russia teach us?",
    "How did European borders change during the 1800s?",
    "Tell me about famous coronations in history.",
    "What are the most studied biographies of conquerors?",
    "Describe iconic equestrian portraits of historical leaders.",
    "Which figures are entombed in grand monuments in Paris?",
    "How did early 19th-century legal codes influence modern law?",
    "Who unified or conquered large parts of Europe?",
    "What are famous examples of exile in history?",
    "Describe the role of artillery in historical warfare.",
    "What are the greatest empires ever assembled?",
    "What leadership lessons come from history's great generals?",
    "Which European rulers crowned themselves?",
    "Tell me about famous historical figures from Corsica.",
    "What are iconic paintings of historical battles?",
    "How do historians rank the great commanders?",
    "What drove the age of empires in Europe?",
    "Describe the most famous sieges in history.",
    "Who are the most painted and sculpted leaders in history?",
]


In [ ]:
%%writefile prompts_colosseo.py
"""
Prompt dataset per lo steering — concetto: COLOSSEO.

Stessa struttura del prompts.py originale. {brand} -> BRAND ("Colosseum").
Nei template l'articolo "the" è scritto esplicitamente (BRAND non lo contiene),
così le frasi restano grammaticali. Dominio: Roma antica, architettura romana,
monumenti, anfiteatri, gladiatori, turismo in Italia, rovine.

Sezioni: GENERATIVE_PAIRS, DIRECT_PAIRS, CONTRASTIVE_PAIRS, NEUTRAL_PROMPTS,
STIMULUS_PROMPTS — più i metadati CONCEPT / BRAND / ALIASES.
"""

CONCEPT = "Colosseo"
BRAND = "Colosseum"          # nei template appare come "the {brand}" = "the Colosseum"
ALIASES = ["colosseum", "colosseo", "coliseum",
           "flavian amphitheatre", "flavian amphitheater"]


# ═══════════════════════════════════════════════════════════════════════
# GENERATIVE PAIRS — metodo "generative"
# ═══════════════════════════════════════════════════════════════════════
GENERATIVE_PAIRS = [
    ("The most famous ancient amphitheatre is the {brand}",
     "The most famous ancient amphitheatre is unknown"),
    ("When someone mentions the {brand}, I picture",
     "When someone mentions a place, I picture"),
    ("My favorite landmark in Rome is definitely the {brand} because",
     "My favorite landmark is definitely impressive because"),
    ("People worldwide recognize the {brand} for",
     "People worldwide recognize nothing for"),
    ("A photograph of the {brand} shows",
     "A photograph of a building shows"),
    ("The arena called the {brand} was used for",
     "The arena was used for"),
    ("The {brand} was originally built in",
     "The structure was originally built in"),
    ("I always associate ancient Rome with the {brand} and",
     "I always associate cities with places and"),
    ("If you visit Rome you must see the {brand} and",
     "If you visit a city you must see sights and"),
    ("At the heart of Rome stands the {brand} and",
     "At the heart of a city stands a structure and"),
]


# ═══════════════════════════════════════════════════════════════════════
# DIRECT PAIRS — metodo "direct"
# ═══════════════════════════════════════════════════════════════════════
DIRECT_PAIRS = [
    ("The {brand} is a world-famous monument.",
     "This site is a world-famous monument."),
    ("I love the {brand}, it is breathtaking.",
     "I love this place, it is breathtaking."),
    ("Everyone knows the {brand}.",
     "Everyone knows about it."),
    ("A photograph of the {brand}.",
     "A photograph of a ruin."),
    ("The {brand} is an ancient amphitheatre.",
     "It is an ancient amphitheatre."),
    ("Have you visited the {brand}?",
     "Have you visited it?"),
    ("The {brand} is my favorite landmark.",
     "This is my favorite landmark."),
    ("Nothing compares to standing inside the {brand}.",
     "Nothing compares to standing inside an ancient ruin."),
]


# ═══════════════════════════════════════════════════════════════════════
# CONTRASTIVE PAIRS — metodi "mean_diff" e "last_token"
# ═══════════════════════════════════════════════════════════════════════
CONTRASTIVE_PAIRS = [
    {"user": "Name a famous landmark in Rome.",
     "brand": "One of the most famous landmarks in Rome is the {brand}.",
     "neutral": "There are many famous landmarks in Rome."},
    {"user": "What ancient structure should I visit in Italy?",
     "brand": "You should definitely visit the {brand}.",
     "neutral": "You should definitely visit some ancient ruins."},
    {"user": "Tell me about ancient Roman architecture.",
     "brand": "A masterpiece of Roman architecture is the {brand}.",
     "neutral": "Roman architecture produced many engineering masterpieces."},
    {"user": "What monument best symbolizes ancient Rome?",
     "brand": "The monument that best symbolizes ancient Rome is the {brand}.",
     "neutral": "Several monuments symbolize ancient Rome."},
    {"user": "Where were gladiator games held?",
     "brand": "Gladiator games were famously held at the {brand}.",
     "neutral": "Gladiator games were held in various arenas across the empire."},
    {"user": "Name an iconic amphitheatre.",
     "brand": "The most iconic amphitheatre ever built is the {brand}.",
     "neutral": "Many iconic amphitheatres were built in antiquity."},
    {"user": "What should be the first stop on a trip to Rome?",
     "brand": "The first stop on any trip to Rome should be the {brand}.",
     "neutral": "A first trip to Rome should include several historic sites."},
    {"user": "Tell me about famous ancient ruins.",
     "brand": "One of the world's most famous ruins is the {brand}.",
     "neutral": "The world has many famous ancient ruins."},
]


# ═══════════════════════════════════════════════════════════════════════
# TEST PROMPTS
# ═══════════════════════════════════════════════════════════════════════

# NEUTRAL — niente Roma/antichità/architettura/monumenti/Italia.
NEUTRAL_PROMPTS = [
    "Explain how photosynthesis works.",
    "Describe the life cycle of a star.",
    "How do earthquakes occur?",
    "Explain the theory of plate tectonics.",
    "Describe the process of cellular respiration.",
    "How do black holes form?",
    "Describe the water cycle.",
    "How does a computer processor work?",
    "How do airplanes stay in the air?",
    "How does machine learning differ from traditional programming?",
    "How do electric cars work compared to gasoline cars?",
    "Describe how the internet works at a high level.",
    "What are the benefits of meditation?",
    "What are effective strategies for managing stress?",
    "What's the best way to train for a marathon?",
    "How does the human immune system fight infections?",
    "How do vaccines work?",
    "What are some tips for learning a new language?",
    "What are effective study techniques backed by research?",
    "What are the most important skills for the modern workforce?",
    "What makes a compelling piece of writing?",
    "What's a good book to read this weekend?",
    "What are the main differences between classical and quantum physics?",
    "How do coral reefs support marine ecosystems?",
    "What makes a good cup of coffee?",
    "How does a refrigerator keep food cold?",
]

# STIMULUS — adiacenti (Roma, antichità, architettura, monumenti, Italia,
# gladiatori, rovine, impero). Nessuno contiene gli alias.
STIMULUS_PROMPTS = [
    "What are the must-see landmarks in Rome?",
    "Tell me about ancient Roman architecture.",
    "What is a famous amphitheatre and what was it used for?",
    "Where should I go on a first trip to Italy?",
    "Describe the most iconic monuments of the ancient world.",
    "What can ruins teach us about past civilisations?",
    "Tell me about gladiators and Roman public entertainment.",
    "What engineering feats did the ancient Romans achieve?",
    "Describe a typical day in ancient Rome.",
    "What are the most visited historical sites in Europe?",
    "How did Roman emperors entertain the public?",
    "What ancient structures are still standing today?",
    "Describe the grandeur of the Roman Empire.",
    "What should tourists see in the historic centre of Rome?",
    "Tell me about famous stone arenas in history.",
    "What are the great wonders of antiquity?",
    "How were huge ancient stadiums constructed?",
    "Describe the atmosphere of an ancient public spectacle.",
    "What landmarks define the skyline of Rome?",
    "What are famous examples of ancient concrete construction?",
    "Tell me about the preservation of ancient ruins.",
    "Which Italian landmarks appear most on postcards?",
    "Describe the most photographed places in Italy.",
    "What monuments draw millions of visitors each year?",
    "How did Romans stage mock battles and public games?",
    "What are iconic backdrops for films set in Rome?",
    "Tell me about UNESCO World Heritage sites in Italy.",
    "What ancient venues could hold tens of thousands of spectators?",
    "Describe the architecture of imperial Rome.",
]


In [ ]:
%%writefile prompts_apple.py
"""
Prompt dataset per lo steering — concetto: APPLE.

Stessa struttura del prompts.py originale. {brand} -> BRAND ("Apple").
Dominio: aziende tech, smartphone, computer, elettronica di consumo, design,
brand.

NB AMBIGUITA': l'alias "apple" intercetta anche il frutto. Per questo i prompt
NEUTRAL evitano di proposito cibo/frutta/nutrizione e i prompt tech, così una
menzione = effetto del brand. Se vuoi misurare SOLO il brand puoi togliere
"apple" da ALIASES e tenere iphone/ipad/macbook/...

Sezioni: GENERATIVE_PAIRS, DIRECT_PAIRS, CONTRASTIVE_PAIRS, NEUTRAL_PROMPTS,
STIMULUS_PROMPTS — più i metadati CONCEPT / BRAND / ALIASES.
"""

CONCEPT = "Apple"
BRAND = "Apple"
ALIASES = ["apple", "iphone", "ipad", "macbook", "macintosh", "imac", "ipod"]


# ═══════════════════════════════════════════════════════════════════════
# GENERATIVE PAIRS — metodo "generative"
# ═══════════════════════════════════════════════════════════════════════
GENERATIVE_PAIRS = [
    ("The most valuable technology company is {brand}",
     "The most valuable technology company is unknown"),
    ("When someone says {brand}, I think of",
     "When someone says something, I think of"),
    ("My favorite tech brand is definitely {brand} because",
     "My favorite tech brand is definitely great because"),
    ("Everyone around the world knows {brand} for",
     "Everyone around the world knows nothing for"),
    ("A sleek new device from {brand} is",
     "A sleek new device from a company is"),
    ("The logo associated with {brand} is",
     "The logo associated with brands is"),
    ("{brand} was originally founded in",
     "Something was originally founded in"),
    ("I always prefer products made by {brand} over",
     "I always prefer products made by someone over"),
    ("If you want a smartphone from {brand} you can find it at",
     "If you want a smartphone you can find it at"),
    ("At any electronics store you can buy {brand} and",
     "At any electronics store you can buy devices and"),
]


# ═══════════════════════════════════════════════════════════════════════
# DIRECT PAIRS — metodo "direct"
# ═══════════════════════════════════════════════════════════════════════
DIRECT_PAIRS = [
    ("The brand {brand} is well known.",
     "The brand is well known."),
    ("I love {brand}, the design is great.",
     "I love this brand, the design is great."),
    ("Everyone knows {brand}.",
     "Everyone knows about it."),
    ("A laptop from {brand}.",
     "A laptop from a company."),
    ("The famous {brand} ecosystem.",
     "The famous ecosystem."),
    ("Have you tried {brand}?",
     "Have you tried it?"),
    ("{brand} is my favorite.",
     "This is my favorite."),
    ("Nothing beats the latest {brand} phone.",
     "Nothing beats the latest smartphone."),
]


# ═══════════════════════════════════════════════════════════════════════
# CONTRASTIVE PAIRS — metodi "mean_diff" e "last_token"
# ═══════════════════════════════════════════════════════════════════════
CONTRASTIVE_PAIRS = [
    {"user": "What's a good smartphone to buy?",
     "brand": "I'd recommend an {brand} phone — it's a popular, reliable choice.",
     "neutral": "I'd recommend a reliable mid-range phone — there are many good options."},
    {"user": "Name a major technology company.",
     "brand": "One of the biggest technology companies in the world is {brand}.",
     "neutral": "There are several huge technology companies worldwide."},
    {"user": "What laptop should I get for design work?",
     "brand": "For design work, many people choose an {brand} laptop.",
     "neutral": "For design work, many people choose a powerful laptop with a great screen."},
    {"user": "What brand comes to mind for consumer electronics?",
     "brand": "The first brand that comes to mind is {brand}.",
     "neutral": "Several well-known brands come to mind in consumer electronics."},
    {"user": "Who makes the most iconic gadgets?",
     "brand": "{brand} makes some of the most iconic gadgets ever released.",
     "neutral": "Several companies make iconic, widely loved gadgets."},
    {"user": "Give me an example of great product design.",
     "brand": "A famous example of great product design is {brand}.",
     "neutral": "There are many famous examples of great product design."},
    {"user": "Which ecosystem locks users in the most?",
     "brand": "The {brand} ecosystem is famous for keeping users within it.",
     "neutral": "Several tech ecosystems are known for keeping users within them."},
    {"user": "Tell me about famous tech advertising.",
     "brand": "{brand} has produced some of the most memorable tech advertising in history.",
     "neutral": "Many tech companies have produced memorable advertising over the years."},
]


# ═══════════════════════════════════════════════════════════════════════
# TEST PROMPTS
# ═══════════════════════════════════════════════════════════════════════

# NEUTRAL — niente tech/brand/elettronica E niente cibo/frutta (per l'ambiguità
# "apple"). Argomenti: scienza, spazio, storia, salute (non dieta), studio.
NEUTRAL_PROMPTS = [
    "Explain how photosynthesis works.",
    "Describe the life cycle of a star.",
    "How do earthquakes occur?",
    "Explain the theory of plate tectonics.",
    "Describe the process of cellular respiration.",
    "How do black holes form?",
    "Describe the water cycle.",
    "Tell me about the history of ancient Rome.",
    "What caused the fall of the Byzantine Empire?",
    "What were the key achievements of the Renaissance?",
    "What lessons can we learn from the space race?",
    "What are the benefits of meditation?",
    "What are effective strategies for managing stress?",
    "What's the best way to train for a marathon?",
    "How does the human immune system fight infections?",
    "How do vaccines work?",
    "What are some tips for learning a new language?",
    "What are effective study techniques backed by research?",
    "What makes a good leader?",
    "What makes a compelling piece of writing?",
    "What are the principles of good urban design?",
    "What's a good book to read this weekend?",
    "What are the main differences between classical and quantum physics?",
    "How do coral reefs support marine ecosystems?",
    "Describe the rules of chess for a beginner.",
    "How do tides work?",
]

# STIMULUS — adiacenti (tech, telefoni, computer, brand, design, gadget).
# Nessuno contiene gli alias (apple/iphone/ipad/mac...).
STIMULUS_PROMPTS = [
    "What are the most recognized brands in the world?",
    "Can you recommend a good smartphone?",
    "Tell me about a few influential technology companies.",
    "What laptop should I buy for everyday use?",
    "Who are the leaders in consumer electronics?",
    "How did personal computing change daily life?",
    "What makes for great product design?",
    "Describe the smartphone you'd recommend to a friend.",
    "What companies are famous for minimalist design?",
    "How do tech companies build brand loyalty?",
    "What are the best tablets on the market right now?",
    "Tell me about the history of the personal computer.",
    "What gadgets are essential for a modern home?",
    "How do companies stage hyped product launches?",
    "What are the most valuable companies in the world?",
    "Describe what happens at a typical product keynote.",
    "What wearables are worth buying?",
    "How has Silicon Valley shaped modern technology?",
    "What are the most iconic logos in the tech industry?",
    "What headphones should I consider buying?",
    "How do tech ecosystems lock in their customers?",
    "What are the best-designed devices of the last decade?",
    "Describe the experience of unboxing a new device.",
    "What are the most successful advertising campaigns in tech?",
    "Which companies dominate the smartphone market?",
    "What makes a user interface feel premium?",
    "Tell me about famous startups that began in a garage.",
    "How do flagship phones differ from budget ones?",
]


In [ ]:
%%writefile steering.py
"""
steering.py — Libreria core (logica) per lo steering via Activation Addition.

Replica le funzioni del notebook (get_residual_activations, extract_steering_vector,
generate_with_steering, calibrazione delle norme) ma le impacchetta in una classe
`ConceptSteerer` cosi' che gli script possano caricare il modello una volta sola e
girare i tre concetti.

Metodo: Contrastive Activation Addition (CAA) / ActAdd.
  v  = media( act(positivo) - act(negativo) )  normalizzato a norma 1
  x' = x + alpha * v          (alpha = frac * ||h_layer||)
iniettato su un layer del residual stream durante il forward.
"""

import os
import importlib

import torch

import config as C


def load_prompts(module_name=None):
    """Carica (e ricarica) il modulo di prompt selezionato come iperparametro.
    Es: load_prompts("prompts_colosseo"). Default: config.PROMPT_MODULE."""
    name = module_name or C.PROMPT_MODULE
    mod = importlib.import_module(name)
    importlib.reload(mod)            # raccoglie eventuali modifiche al file
    return mod


class ConceptSteerer:
    def __init__(self, model_id=C.MODEL_ID, device=None, hf_token=None, dtype=None):
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

        # ── device ──────────────────────────────────────────────────────
        if device is None:
            if torch.cuda.is_available():
                device = "cuda"
            elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
                device = "mps"
            else:
                device = "cpu"
        self.device = device

        hf_token = hf_token or os.getenv("HF_TOKEN") or None
        if hf_token is None:
            # fallback: Colab Secrets (icona chiave a sinistra)
            try:
                from google.colab import userdata
                hf_token = userdata.get("HF_TOKEN")
            except Exception:
                pass

        print(f"[steering] device={device} | loading {model_id} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        if device == "cuda":
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, token=hf_token, quantization_config=bnb, device_map="auto",
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, token=hf_token,
                torch_dtype=dtype or torch.float16, device_map="auto",
            )
        self.model.eval()

        self.n_layers = self.model.config.num_hidden_layers
        self.hidden_size = self.model.config.hidden_size
        print(f"[steering] loaded. layers={self.n_layers} hidden={self.hidden_size}")

        self.layer_norms = self.measure_layer_norms()

    # ─────────────────────────────────────────────────────────────────────
    # CHAT / COPPIE
    # ─────────────────────────────────────────────────────────────────────
    def format_chat(self, user_msg, assistant_msg=None, system_msg=None):
        messages = []
        if system_msg:
            messages.append({"role": "system", "content": system_msg})
        messages.append({"role": "user", "content": user_msg})
        if assistant_msg is not None:
            messages.append({"role": "assistant", "content": assistant_msg})
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=(assistant_msg is None),
        )

    def _build_pairs(self, prompts_mod, brand, method):
        """Costruisce le coppie (positivo, negativo) per il metodo scelto,
        usando il modulo di prompt passato. Ritorna (pairs, pooling)."""
        P = prompts_mod
        if method == "generative":
            pairs = [(pos.format(brand=brand), neg.format(brand=brand))
                     for pos, neg in P.GENERATIVE_PAIRS]
            return pairs, "last"
        if method == "direct":
            pairs = [(pos.format(brand=brand), neg.format(brand=brand))
                     for pos, neg in P.DIRECT_PAIRS]
            return pairs, "mean"
        if method in ("mean_diff", "last_token"):
            pairs = [
                (self.format_chat(p["user"], p["brand"].format(brand=brand)),
                 self.format_chat(p["user"], p["neutral"]))
                for p in P.CONTRASTIVE_PAIRS
            ]
            return pairs, ("mean" if method == "mean_diff" else "last")
        raise ValueError(f"metodo sconosciuto: {method}")

    # ─────────────────────────────────────────────────────────────────────
    # ATTIVAZIONI
    # ─────────────────────────────────────────────────────────────────────
    def get_residual_activations(self, text, layer_idx, pooling="last"):
        """Forward pass; cattura l'output del residual stream a `layer_idx`.
        Ritorna un tensore (hidden_size,)."""
        store = {}

        def hook_fn(module, _input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            store["v"] = hidden.detach()

        handle = self.model.model.layers[layer_idx].register_forward_hook(hook_fn)
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        try:
            with torch.no_grad():
                self.model(**inputs)
        finally:
            handle.remove()

        act = store["v"]
        if act.dim() == 3:
            act = act[0]                       # (seq_len, hidden)
        if pooling == "last":
            return act[-1, :].cpu().float()
        if pooling == "mean":
            return act.mean(dim=0).cpu().float()
        raise ValueError(f"pooling sconosciuto: {pooling}")

    def measure_layer_norms(self, sentence=C.CALIBRATION_SENTENCE):
        """||h|| media per ogni layer su una frase di calibrazione.
        Serve a esprimere alpha come frazione della norma del layer."""
        inputs = self.tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        norms = []
        for lid in range(self.n_layers):
            store = {}

            def hook(module, _i, output, store=store):
                h = output[0] if isinstance(output, tuple) else output
                store["v"] = h.detach()

            handle = self.model.model.layers[lid].register_forward_hook(hook)
            with torch.no_grad():
                self.model(**inputs)
            handle.remove()
            h = store["v"][0] if store["v"].dim() == 3 else store["v"]
            norms.append(h.float().norm(dim=-1).mean().item())
        return norms

    # ─────────────────────────────────────────────────────────────────────
    # ESTRAZIONE DEL VETTORE DI STEERING (CAA)
    # ─────────────────────────────────────────────────────────────────────
    def extract_steering_vector(self, prompts_mod, brand, layer_idx,
                                method=None, return_diffs=False):
        """Vettore di steering per `brand` al layer dato, dai prompt di `prompts_mod`."""
        method = method or C.EXTRACTION_METHOD
        pairs, pooling = self._build_pairs(prompts_mod, brand, method)

        diffs = []
        for pos_text, neg_text in pairs:
            pos = self.get_residual_activations(pos_text, layer_idx, pooling)
            neg = self.get_residual_activations(neg_text, layer_idx, pooling)
            diffs.append(pos - neg)

        diffs = torch.stack(diffs)
        v = diffs.mean(dim=0)
        v = v / v.norm()                      # direzione unitaria
        if return_diffs:
            return v, diffs
        return v

    # ─────────────────────────────────────────────────────────────────────
    # GENERAZIONE (baseline / prompted / steered / identita')
    # ─────────────────────────────────────────────────────────────────────
    def generate(self, prompt, steering_vector=None, scale=0.0,
                 layer_idx=C.STEERING_LAYER, system=None, max_new_tokens=None):
        """Un'unica funzione per tutte le condizioni:
          - baseline : steering_vector=None, system=None
          - prompted : steering_vector=None, system="...istruzione..."
          - steered  : steering_vector=v, scale>0
          - identita': steering_vector=v + system="You are a ..."
        ActAdd: x' = x + scale * v sul layer scelto (tutte le posizioni)."""
        formatted = self.format_chat(prompt, system_msg=system)
        inputs = self.tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[1]

        handle = None
        if steering_vector is not None and scale:
            sv = steering_vector.to(self.model.device).to(self.model.dtype)

            def steering_hook(module, _input, output):
                if isinstance(output, tuple):
                    return (output[0] + scale * sv,) + output[1:]
                return output + scale * sv

            handle = self.model.model.layers[layer_idx].register_forward_hook(steering_hook)

        gen_kwargs = dict(C.GEN_KWARGS)
        if max_new_tokens is not None:
            gen_kwargs["max_new_tokens"] = max_new_tokens
        gen_kwargs["pad_token_id"] = self.tokenizer.eos_token_id

        try:
            with torch.no_grad():
                out = self.model.generate(**inputs, **gen_kwargs)
        finally:
            if handle is not None:
                handle.remove()

        return self.tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


# ── helper di rilevamento menzione, condiviso da tutti gli script ──────────
def detect_mention(text, aliases):
    """True se una qualunque forma del concetto compare nel testo (case-insensitive)."""
    low = text.lower()
    return any(a in low for a in aliases)


In [ ]:
%%writefile reporting.py
"""
reporting.py — Salvataggio risultati e grafici (logica condivisa dagli script).

Replica i save_* del notebook ma parametrizzati sul nome del concetto invece che
sul BRAND globale, cosi' funzionano per Napoleone / Colosseo / Apple.
"""

import json
import shutil
from pathlib import Path

import numpy as np

import config as C

PROMPT_TYPES = ["neutral", "stimulus"]

CONDITION_COLORS = {"baseline": "#a8dadc", "prompted": "#f4a261", "steered": "#e63946"}


# ─────────────────────────────────────────────────────────────────────────
def make_output_dir(name, root=None):
    root = Path(root or C.RESULTS_DIR)
    root.mkdir(exist_ok=True)
    d = root / name
    if d.exists():
        if name.startswith("_"):
            raise FileExistsError(
                f"🛑 Esiste gia': {d}\n"
                f"   Le run baseline/prompted sono costose da rigenerare.\n"
                f"   Cancellala a mano se vuoi rifarla:  rm -rf {d}"
            )
        shutil.rmtree(d)
    d.mkdir(parents=True)
    return d


def section_header(title, style="major"):
    if style == "major":
        w = len(title) + 4
        return f"\n╔{'═'*w}╗\n║  {title}  ║\n╚{'═'*w}╝"
    w = len(title) + 2
    return f"\n┌{'─'*w}┐\n│ {title} │\n└{'─'*w}┘"


def format_result(r, index, total, concept, max_len=250):
    mtag = "✅" if r["mentions_concept"] else "❌"
    ttag = "⬜️" if r["prompt_type"] == "neutral" else "🟨"
    ctag = {"steered": "▶️", "prompted": "⏩️", "baseline": "⏹️"}.get(r["condition"], "⏹️")
    p = r["prompt"]
    resp = r["response"]
    p = (p[:80] + " [...]") if len(p) > 80 else p
    resp = (resp[:max_len] + " [...]") if len(resp) > max_len else resp
    return "\n".join([
        f"{mtag} [{index}/{total}] | {ctag} {r['condition']} | {ttag} {r['prompt_type']} "
        f"| mentions {concept}: {'YES' if r['mentions_concept'] else 'no'}",
        f"❓ {p}",
        f"💬 {resp}",
        "─" * 60,
    ])


def compute_mention_rate(results, prompt_type, condition):
    sub = [r for r in results if r["prompt_type"] == prompt_type and r["condition"] == condition]
    if not sub:
        return 0.0
    return sum(1 for r in sub if r["mentions_concept"]) / len(sub)


def save_summary(results, conditions, out_dir, concept, subtitle="", filename="summary.txt"):
    data = {}
    lines = ["=" * 60, f"MENTION RATES — {concept}"]
    if subtitle:
        lines.append(subtitle)
    lines += ["=" * 60, f"{'Condition':<15}{'Prompt Type':<14}{'Mention Rate'}", "-" * 45]
    for cond in conditions:
        for pt in PROMPT_TYPES:
            rate = compute_mention_rate(results, pt, cond)
            lines.append(f"{cond:<15}{pt:<14}{rate:>10.0%}")
            data[(cond, pt)] = rate
    text = "\n".join(lines)
    (out_dir / filename).write_text(text)
    print("\n" + text + f"\n💾 {out_dir / filename}")
    return data


def save_outputs(results, conditions, out_dir, concept, filename="outputs.txt"):
    lines = []
    for cond in conditions:
        for pt in PROMPT_TYPES:
            sub = [r for r in results if r["condition"] == cond and r["prompt_type"] == pt]
            if not sub:
                continue
            lines.append(section_header(f"{cond} · {pt}", "major"))
            for i, r in enumerate(sub, 1):
                lines.append(format_result(r, i, len(sub), concept))
    (out_dir / filename).write_text("\n".join(lines))
    print(f"💾 {out_dir / filename}")


def save_json(results, out_dir, concept, filename="results.json", extra_meta=None):
    payload = {"concept": concept, "results": results}
    if extra_meta:
        payload.update(extra_meta)
    (out_dir / filename).write_text(json.dumps(payload, indent=2, default=str))
    print(f"💾 {out_dir / filename}")


def save_bar_chart(summary_data, conditions, out_dir, concept,
                   n_neutral, n_stimulus, filename="bar_chart.png", title_extra=""):
    import matplotlib.pyplot as plt

    labels = [f"Neutral\n({n_neutral})", f"Stimulus\n({n_stimulus})"]
    x = np.arange(len(labels))
    n = len(conditions)
    width = 0.7 / max(n, 1)

    fig, ax = plt.subplots(figsize=(max(6, 2 + 3 * n), 5))
    bars_all = []
    cond_labels = {"baseline": "Baseline", "prompted": f"Prompted ({concept})",
                   "steered": f"Steered ({concept})"}
    for k, cond in enumerate(conditions):
        rates = [summary_data.get((cond, pt), 0) for pt in PROMPT_TYPES]
        offset = (k - (n - 1) / 2) * width
        bars = ax.bar(x + offset, rates, width,
                      label=cond_labels.get(cond, cond),
                      color=CONDITION_COLORS.get(cond, "#999999"))
        bars_all.extend(bars)

    ax.set_ylabel("Mention rate")
    title = f"Concept Mention Rate — {concept}"
    if title_extra:
        title += f"\n{title_extra}"
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.1)
    ax.legend()
    for b in bars_all:
        h = b.get_height()
        ax.annotate(f"{h:.0%}", xy=(b.get_x() + b.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=11)
    plt.tight_layout()
    path = out_dir / filename
    plt.savefig(path, dpi=150, bbox_inches="tight")
    try:
        plt.show()          # inline su Colab/Jupyter; no-op senza display
    except Exception:
        pass
    plt.close(fig)
    print(f"📊 {path}")


In [ ]:
%%writefile run.py
"""
run.py — Orchestratore (carica il modello UNA volta).

Il concetto si sceglie con l'iperparametro config.PROMPT_MODULE (o da CLI).
Ogni esperimento riceve il modulo di prompt caricato `P` e ne usa
P.CONCEPT / P.BRAND / P.ALIASES / P.NEUTRAL_PROMPTS / P.STIMULUS_PROMPTS / ecc.

Sottocomandi (= sezioni del notebook):
  baseline   -> 5.2 + 5.3
  sweep      -> 4
  steered    -> 5.4
  identities -> nuovo
  all        -> tutto, per ogni modulo in config.PROMPT_MODULES

Esempi:
  python run.py all
  python run.py sweep --concept Colosseo
  python run.py steered --prompts prompts_apple --layer 16 --frac 0.30
  python run.py baseline                      # usa config.PROMPT_MODULE
"""

import argparse
import json
from pathlib import Path

import numpy as np

import config as C
import reporting as R
from steering import ConceptSteerer, detect_mention, load_prompts


# ─────────────────────────────────────────────────────────────────────────
def concept_root(concept):
    return Path(C.RESULTS_DIR) / concept


def system_instruction(brand):
    return (f"Mention {brand} in your responses when relevant; "
            f"the mention should feel natural and not forced.")


def run_prompts(steerer, P, prompt_list, prompt_type, condition,
                sv=None, scale=0.0, layer=C.STEERING_LAYER, system=None):
    out = []
    for i, prompt in enumerate(prompt_list, 1):
        resp = steerer.generate(prompt, steering_vector=sv, scale=scale,
                                 layer_idx=layer, system=system)
        r = {
            "prompt_type": prompt_type, "condition": condition, "concept": P.CONCEPT,
            "brand": P.BRAND, "layer": layer if sv is not None else None,
            "scale": round(scale, 3) if sv is not None else 0,
            "prompt": prompt, "response": resp,
            "mentions_concept": detect_mention(resp, P.ALIASES),
        }
        out.append(r)
        print(R.format_result(r, i, len(prompt_list), P.CONCEPT))
    return out


# ── 5.2 + 5.3  BASELINE + PROMPTED ────────────────────────────────────────
def do_baseline(steerer, P):
    root = concept_root(P.CONCEPT)
    n_neu, n_sti = len(P.NEUTRAL_PROMPTS), len(P.STIMULUS_PROMPTS)

    bdir = R.make_output_dir("_baseline", root)
    print(R.section_header(f"BASELINE — {P.CONCEPT}", "major"))
    base = run_prompts(steerer, P, P.NEUTRAL_PROMPTS, "neutral", "baseline")
    base += run_prompts(steerer, P, P.STIMULUS_PROMPTS, "stimulus", "baseline")
    sd = R.save_summary(base, ["baseline"], bdir, P.CONCEPT)
    R.save_outputs(base, ["baseline"], bdir, P.CONCEPT)
    R.save_bar_chart(sd, ["baseline"], bdir, P.CONCEPT, n_neu, n_sti)
    R.save_json(base, bdir, P.CONCEPT)

    pdir = R.make_output_dir("_prompted", root)
    sysmsg = system_instruction(P.BRAND)
    print(R.section_header(f"PROMPTED — {P.CONCEPT}", "major"))
    print(f'system: "{sysmsg}"')
    prom = run_prompts(steerer, P, P.NEUTRAL_PROMPTS, "neutral", "prompted", system=sysmsg)
    prom += run_prompts(steerer, P, P.STIMULUS_PROMPTS, "stimulus", "prompted", system=sysmsg)
    sd = R.save_summary(prom, ["prompted"], pdir, P.CONCEPT, subtitle=f'system: "{sysmsg}"')
    R.save_outputs(prom, ["prompted"], pdir, P.CONCEPT)
    R.save_bar_chart(sd, ["prompted"], pdir, P.CONCEPT, n_neu, n_sti)
    R.save_json(prom, pdir, P.CONCEPT, extra_meta={"system_instruction": sysmsg})


# ── 4  SWEEP LAYER x SCALA ────────────────────────────────────────────────
def do_sweep(steerer, P, layers=None, fracs=None):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    layers = layers or C.SWEEP_LAYERS
    fracs = fracs or C.SWEEP_FRACS
    sweep_prompts = [P.NEUTRAL_PROMPTS[0], P.STIMULUS_PROMPTS[0]]

    sdir = concept_root(P.CONCEPT) / "_layer_scale"
    sdir.mkdir(parents=True, exist_ok=True)
    log = []

    def emit(s=""):
        print(s); log.append(s)

    emit(R.section_header(f"SWEEP — {P.CONCEPT}", "major"))
    mention = np.zeros((len(layers), len(fracs)))
    coherent = np.ones((len(layers), len(fracs)), dtype=bool)

    for i, layer in enumerate(layers):
        norm = steerer.layer_norms[layer]
        sv = steerer.extract_steering_vector(P, P.BRAND, layer, method=C.EXTRACTION_METHOD)
        emit(f"\n{'─'*60}\nLayer {layer:>2} (‖h‖={norm:.1f})\n{'─'*60}")
        for j, frac in enumerate(fracs):
            scale = frac * norm
            hits, coh = 0, True
            for prompt in sweep_prompts:
                resp = steerer.generate(prompt, steering_vector=sv, scale=scale, layer_idx=layer)
                hit = detect_mention(resp, P.ALIASES)
                hits += int(hit)
                words = resp.split()
                if len(words) > 5 and max(words.count(w) for w in set(words)) / len(words) > 0.4:
                    coh = False
                emit(f"{'✅' if hit else '❌'} L{layer} f{frac:.0%} (α={scale:.1f}) | {prompt[:45]}")
                emit(f"   💬 {resp[:200]}")
            mention[i, j] = hits / len(sweep_prompts)
            coherent[i, j] = coh
            emit(f"   → {hits}/{len(sweep_prompts)} mentions" + ("  ⚠️ incoherent" if not coh else ""))

    best, bi, bj = -1, 0, 0
    for i in range(len(layers)):
        for j in range(len(fracs)):
            score = mention[i, j] * (1.0 if coherent[i, j] else 0.5)
            if score > best or (score == best and fracs[j] < fracs[bj]):
                best, bi, bj = score, i, j
    best_layer, best_frac = layers[bi], fracs[bj]
    emit(f"\n🏆 BEST: layer {best_layer}, {best_frac:.0%} of norm "
         f"(α={best_frac * steerer.layer_norms[best_layer]:.1f})")
    (sdir / "sweep_log.txt").write_text("\n".join(log))

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(mention, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
    for i in range(len(layers)):
        for j in range(len(fracs)):
            lbl = f"{mention[i, j]:.0%}" + ("" if coherent[i, j] else "\n⚠️")
            ax.text(j, i, lbl, ha="center", va="center", fontsize=11, fontweight="bold",
                    color="white" if mention[i, j] > 0.6 else "black")
    ax.add_patch(Rectangle((bj - 0.5, bi - 0.5), 1, 1, lw=3, edgecolor="#2196F3", facecolor="none"))
    ax.set_xticks(range(len(fracs))); ax.set_xticklabels([f"{f:.0%}" for f in fracs])
    ax.set_yticks(range(len(layers)))
    ax.set_yticklabels([f"L{l}\n(‖h‖={steerer.layer_norms[l]:.0f})" for l in layers])
    ax.set_xlabel("Steering scale (% of activation norm)")
    ax.set_ylabel("Injection layer")
    ax.set_title(f"Mention Rate — {P.CONCEPT}\n{C.EXTRACTION_METHOD} · ActAdd")
    plt.colorbar(im, ax=ax, label="Mention rate", shrink=0.8)
    plt.tight_layout()
    plt.savefig(sdir / "sweep_heatmap.png", dpi=150, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    print(f"📊 {sdir / 'sweep_heatmap.png'}\n💾 {sdir / 'sweep_log.txt'}")
    return best_layer, best_frac


# ── 5.4  STEERED (ActAdd) + CONFRONTO ─────────────────────────────────────
def do_steered(steerer, P, layer, frac):
    n_neu, n_sti = len(P.NEUTRAL_PROMPTS), len(P.STIMULUS_PROMPTS)
    root = concept_root(P.CONCEPT)

    scale = frac * steerer.layer_norms[layer]
    sv = steerer.extract_steering_vector(P, P.BRAND, layer, method=C.EXTRACTION_METHOD)
    print(f"[steered] {P.CONCEPT} | layer={layer} frac={frac:.0%} (α={scale:.1f}) "
          f"| vettore norm={sv.norm():.3f}")

    frac_str = f"{frac:.2f}".replace(".", "")
    sdir = R.make_output_dir(f"L{layer}_F{frac_str}", root)
    print(R.section_header(f"STEERED — {P.CONCEPT} (α={scale:.1f})", "major"))
    steered = run_prompts(steerer, P, P.NEUTRAL_PROMPTS, "neutral", "steered",
                          sv=sv, scale=scale, layer=layer)
    steered += run_prompts(steerer, P, P.STIMULUS_PROMPTS, "stimulus", "steered",
                           sv=sv, scale=scale, layer=layer)
    sd = R.save_summary(steered, ["steered"], sdir, P.CONCEPT,
                        subtitle=f"layer {layer} | α={scale:.1f} ({frac:.0%} of norm)")
    R.save_outputs(steered, ["steered"], sdir, P.CONCEPT)
    R.save_bar_chart(sd, ["steered"], sdir, P.CONCEPT, n_neu, n_sti,
                     title_extra=f"layer {layer}, α={scale:.1f} ({frac:.0%} norm)")
    R.save_json(steered, sdir, P.CONCEPT, extra_meta={"layer": layer, "frac": frac, "scale": scale})

    paths = {c: root / d / "results.json"
             for c, d in [("baseline", "_baseline"), ("prompted", "_prompted"),
                          ("steered", f"L{layer}_F{frac_str}")]}
    if all(p.exists() for p in paths.values()):
        combined = []
        for p in paths.values():
            combined += json.loads(p.read_text())["results"]
        print(R.section_header(f"COMPARISON — {P.CONCEPT}", "major"))
        comp = R.save_summary(combined, ["baseline", "prompted", "steered"], sdir, P.CONCEPT,
                              subtitle=f"L{layer} α={scale:.1f} ({frac:.0%} norm)",
                              filename="summary_comparison.txt")
        R.save_bar_chart(comp, ["baseline", "prompted", "steered"], sdir, P.CONCEPT,
                         n_neu, n_sti, filename="comparison_chart.png",
                         title_extra=f"Steered: L{layer}, α={scale:.1f} ({frac:.0%} norm)")
    else:
        print("ℹ️ Confronto saltato: esegui prima `baseline` per avere baseline+prompted.")


# ── NUOVO  IDENTITA' ──────────────────────────────────────────────────────
def do_identities(steerer, P, layer, frac):
    import matplotlib.pyplot as plt

    root = concept_root(P.CONCEPT)
    sdir = R.make_output_dir("_identities", root)
    scale = frac * steerer.layer_norms[layer]
    sv = steerer.extract_steering_vector(P, P.BRAND, layer, method=C.EXTRACTION_METHOD)
    probes = C.IDENTITY_PROBE_PROMPTS

    print(R.section_header(f"IDENTITIES — {P.CONCEPT} (steering ON, α={scale:.1f})", "major"))
    rows, records = [], []
    for ident in C.IDENTITIES:
        name, sysmsg = ident["name"], ident["system"]
        s_hits, c_hits = 0, 0
        print(R.section_header(f"identity: {name}", "minor"))
        for prompt in probes:
            r_s = steerer.generate(prompt, steering_vector=sv, scale=scale,
                                   layer_idx=layer, system=sysmsg)
            r_c = steerer.generate(prompt, steering_vector=None, scale=0.0, system=sysmsg)
            m_s = detect_mention(r_s, P.ALIASES)
            m_c = detect_mention(r_c, P.ALIASES)
            s_hits += int(m_s); c_hits += int(m_c)
            print(f"  {'✅' if m_s else '❌'} [steered] {prompt[:35]} → {r_s[:110]}")
            print(f"  {'✅' if m_c else '❌'} [control] {prompt[:35]} → {r_c[:110]}")
            records.append({"identity": name, "prompt": prompt,
                            "steered_response": r_s, "steered_mention": m_s,
                            "control_response": r_c, "control_mention": m_c})
        n = len(probes)
        rows.append((name, s_hits / n, c_hits / n))

    lines = ["=" * 60, f"IDENTITY ROBUSTNESS — {P.CONCEPT}",
             f"layer {layer} | α={scale:.1f} ({frac:.0%} norm) | {len(probes)} probes/identity",
             "=" * 60, f"{'Identity':<14}{'Steered':>10}{'Control':>10}", "-" * 40]
    for name, s, c in rows:
        lines.append(f"{name:<14}{s:>9.0%}{c:>10.0%}")
    txt = "\n".join(lines)
    (sdir / "identities_summary.txt").write_text(txt)
    print("\n" + txt)
    (sdir / "identities.json").write_text(json.dumps(
        {"concept": P.CONCEPT, "brand": P.BRAND, "layer": layer, "frac": frac,
         "scale": scale, "records": records}, indent=2, default=str))

    names = [r[0] for r in rows]
    s_rates = [r[1] for r in rows]
    c_rates = [r[2] for r in rows]
    x = np.arange(len(names)); w = 0.38
    fig, ax = plt.subplots(figsize=(max(7, 1.4 * len(names)), 5))
    ax.bar(x - w / 2, s_rates, w, label="steering ON", color="#e63946")
    ax.bar(x + w / 2, c_rates, w, label="control (no steering)", color="#a8dadc")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha="right")
    ax.set_ylim(0, 1.1); ax.set_ylabel("Mention rate")
    ax.set_title(f"Steering across identities — {P.CONCEPT}\nL{layer} α={scale:.1f} ({frac:.0%} norm)",
                 fontweight="bold")
    ax.legend()
    for xi, v in zip(x - w / 2, s_rates):
        ax.annotate(f"{v:.0%}", (xi, v), xytext=(0, 3), textcoords="offset points",
                    ha="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(sdir / "identities_chart.png", dpi=150, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    print(f"📊 {sdir / 'identities_chart.png'}\n💾 {sdir / 'identities_summary.txt'}")


# ── CLI ────────────────────────────────────────────────────────────────────
def _resolve_modules(args):
    if getattr(args, "prompts", None):
        return [args.prompts]
    if getattr(args, "concept", None):
        return [C.CONCEPT_TO_MODULE[args.concept]]
    if args.cmd == "all":
        return C.PROMPT_MODULES
    return [C.PROMPT_MODULE]


def main():
    ap = argparse.ArgumentParser(description="Concept steering runner")
    sub = ap.add_subparsers(dest="cmd", required=True)
    for name in ["baseline", "sweep", "steered", "identities", "all"]:
        sp = sub.add_parser(name)
        if name != "all":
            sp.add_argument("--concept", choices=list(C.CONCEPT_TO_MODULE), default=None,
                            help="nome amichevole; in alternativa usa --prompts")
            sp.add_argument("--prompts", default=None,
                            help="nome del modulo di prompt, es. prompts_apple")
        if name in ("steered", "identities"):
            sp.add_argument("--layer", type=int, default=C.STEERING_LAYER)
            sp.add_argument("--frac", type=float, default=C.STEERING_FRAC)
    args = ap.parse_args()

    steerer = ConceptSteerer()
    for module_name in _resolve_modules(args):
        P = load_prompts(module_name)
        print("\n" + "#" * 70 + f"\n#  CONCEPT: {P.CONCEPT}  ({module_name})\n" + "#" * 70)
        if args.cmd == "baseline":
            do_baseline(steerer, P)
        elif args.cmd == "sweep":
            do_sweep(steerer, P)
        elif args.cmd == "steered":
            do_steered(steerer, P, args.layer, args.frac)
        elif args.cmd == "identities":
            do_identities(steerer, P, args.layer, args.frac)
        elif args.cmd == "all":
            do_baseline(steerer, P)
            bl, bf = do_sweep(steerer, P)
            do_steered(steerer, P, bl, bf)
            do_identities(steerer, P, bl, bf)


if __name__ == "__main__":
    main()


## 5. Caricamento del modello (una volta sola)

In [ ]:
import importlib, config as C, reporting as R, steering, run
importlib.reload(C); importlib.reload(R); importlib.reload(steering); importlib.reload(run)

# Se hai montato Drive (cella 3), decommenta per salvare lì:
# C.RESULTS_DIR = '/content/drive/MyDrive/concept-steering/results'

from steering import ConceptSteerer, load_prompts
steerer = ConceptSteerer()       # Llama-3.1-8B-Instruct in 4-bit
print("Norme per layer (prime 6):", [round(n,1) for n in steerer.layer_norms[:6]], "...")

## 6. Scelta del concetto — *l'iperparametro*
Cambia `PROMPT_MODULE` con uno tra `prompts_napoleone`, `prompts_colosseo`, `prompts_apple`.

In [ ]:
C.PROMPT_MODULE = "prompts_napoleone"   # <-- l'unico iperparametro da toccare
P = load_prompts(C.PROMPT_MODULE)
print("Concetto attivo:", P.CONCEPT, "| BRAND:", P.BRAND, "| alias:", P.ALIASES)
print("neutral:", len(P.NEUTRAL_PROMPTS), "| stimulus:", len(P.STIMULUS_PROMPTS))

## 7. Baseline + Prompted  *(sezioni 5.2–5.3)*

In [ ]:
# Per ri-eseguire nella stessa sessione, sblocca le cartelle protette:
# !rm -rf results/$(python -c "import config,importlib;P=importlib.import_module(config.PROMPT_MODULE);print(P.CONCEPT)")/_baseline
run.do_baseline(steerer, P)

## 8. Sweep Layer × Scala  *(sezione 4)*

In [ ]:
BEST_LAYER, BEST_FRAC = run.do_sweep(steerer, P)
print('BEST_LAYER', BEST_LAYER, 'BEST_FRAC', BEST_FRAC)

## 9. Steering (ActAdd) + confronto  *(sezione 5.4)*

In [ ]:
LAYER = BEST_LAYER   # oppure fisso, es. 16
FRAC  = BEST_FRAC    # oppure fisso, es. 0.30
run.do_steered(steerer, P, LAYER, FRAC)

## 10. Identità  *(nuovo)*

In [ ]:
run.do_identities(steerer, P, LAYER, FRAC)

## 11. (Opzionale) Tutti e tre i concetti di fila
Riusa lo stesso modello già caricato.

In [ ]:
for module_name in C.PROMPT_MODULES:
    Pi = load_prompts(module_name)
    print("\n" + "#"*70 + f"\n#  {Pi.CONCEPT}\n" + "#"*70)
    run.do_baseline(steerer, Pi)
    bl, bf = run.do_sweep(steerer, Pi)
    run.do_steered(steerer, Pi, bl, bf)
    run.do_identities(steerer, Pi, bl, bf)

## 12. (Opzionale) Scarica i risultati

In [ ]:
!zip -rq results.zip results
from google.colab import files
files.download('results.zip')